# 00 Schema — the contract and the tools

No content. Checks:

1. every base module of `schema/cdisc_for_all.yaml` follows the mapping rule (class and property IRIs, one string per column, the row key among the columns);
2. generates, per base module, the CSVW table descriptions (`schema/csvw/`) and the structural SHACL (`schema/shacl/`, LinkML gen-shacl);
2c. generates the vocabulary graph (`schema/vocabulary.ttl`, plain RDFS): one class per table, one property per column with its published name as label, one property per link with its description and rule;
3. the tools of `cfa`, on small test fixtures (not source data): the xlsx text round trip, the CSVW conversion, the canonical N-Triples against a canonicalisation library, SHACL catching a fault.

Content checks (graph against source) are in the base readers and the release check.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from linkml_runtime.utils.schemaview import SchemaView
from pyld import jsonld
from rdflib import Graph, Literal, URIRef

from cfa import SCHEMA_DIR
from cfa.graph import canonical_lines, graph_sha256
from cfa.read import read_xlsx_table
from cfa.rule import check_module, csvw_metadata, load_module, module_source, pct, table_to_graph
from cfa.view import write_view

## 1. The base modules follow the mapping rule

The index schema imports every module. A base module carries the annotation `source` (the source slug).

In [ ]:
index = SchemaView(str(SCHEMA_DIR / 'cdisc_for_all.yaml'))
base_modules = []
for name in index.imports_closure():
    path = SCHEMA_DIR / f'{name}.yaml'
    if not path.exists():
        continue
    sv = load_module(path)
    if sv.schema.annotations and 'source' in sv.schema.annotations:
        check_module(sv)
        base_modules.append(path)
print('base modules:', [p.name for p in base_modules])

## 2. Generate the CSVW table descriptions and the structural SHACL

CSVW `url`: for an xlsx source, the table as `cfa.read.read_xlsx_table` reads it (`<table>.csv`, percent-encoded); for a CSV/TSV source, the file (a later step). One CSVW file per source, a table group if the source has more than one table.

In [ ]:
for path in base_modules:
    sv = load_module(path)
    source = module_source(sv)
    tables = list(sv.all_classes())
    descriptions = [csvw_metadata(sv, t, f'{pct(t)}.csv') for t in tables]
    if len(descriptions) == 1:
        meta = descriptions[0]
    else:
        meta = {'@context': descriptions[0]['@context'],
                'tables': [{k: v for k, v in d.items() if k != '@context'} for d in descriptions]}
    out = SCHEMA_DIR / 'csvw' / f'{source}.csv-metadata.json'
    out.write_text(json.dumps(meta, indent=1, ensure_ascii=False) + '\n', encoding='utf-8')
    shacl = SCHEMA_DIR / 'shacl' / f'{source}.shacl.ttl'
    shacl.write_text(ShaclGenerator(str(path)).serialize(), encoding='utf-8')
    print(source, '->', out.relative_to(REPO_ROOT), shacl.relative_to(REPO_ROOT))

## 2b. Check the links and generate the link shapes

`schema/links.yaml` (imported by the index schema): every link names columns of its domain and range classes, its IRI is the one the rule gives (`link/` for a published key, `rule/` for a link by rule), a link by rule states its rule. The link shapes (SHACL-SPARQL, sh:Info) are generated from it.

In [ ]:
from cfa.graph import write_longturtle
from cfa.link import link_shapes, load_links

link_specs = load_links(SCHEMA_DIR / 'links.yaml')
out = SCHEMA_DIR / 'shacl' / 'links.shacl.ttl'
write_longturtle(link_shapes(link_specs), out, {'sh': 'http://www.w3.org/ns/shacl#'})
print(len(link_specs), 'links ->', out.relative_to(REPO_ROOT))

## 2c. Generate the vocabulary graph

`schema/vocabulary.ttl`, in plain RDFS, restated from the schema (`cfa.vocabulary`): one `rdfs:Class` per base table under its class IRI (label: the table name as published); one `rdf:Property` per published column under its property IRI (label: the column name as published, domain: its table, range: xsd:string); one `rdf:Property` per link of links.yaml (label: the link name, domain and range: the tables it joins, its kind, description, rule and marker rule as stated). Each term names its LinkML module (`rdfs:isDefinedBy`). LinkML's gen-owl is not used: it names properties by its own rule, not by the slot_uri the graphs use. `linked/Make_Catalog.ipynb` checks the vocabulary against the built graphs.

In [ ]:
from cfa.vocabulary import vocabulary, vocabulary_prefixes

vocab = vocabulary(SCHEMA_DIR / 'cdisc_for_all.yaml', link_specs)
out = SCHEMA_DIR / 'vocabulary.ttl'
write_longturtle(vocab, out, vocabulary_prefixes(vocab))
assert graph_sha256(Graph().parse(out)) == graph_sha256(vocab)
print(len(vocab), 'triples ->', out.relative_to(REPO_ROOT))

## 3. The tools, on test fixtures

### 3a. xlsx text round trip

Texts that a spreadsheet tends to change: an escape-shaped text, CR+LF, surrounding whitespace and tab, a formula-shaped text, leading zeros. Written by `cfa.view.write_view`, read back by `cfa.read.read_xlsx_table`: must be unchanged.

In [ ]:
texts = ['_x000D_', 'a\r\nb', ' lead', 'trail\t', '=1+1', '00123', 'x_x005F_y']
fixture = pd.DataFrame({'k': [str(i) for i in range(len(texts))], 'text': texts}, dtype=str)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'fixture.xlsx'
    schema_df = pd.DataFrame(columns=['Sheet'], dtype=str)
    write_view(path, ['fixture'], {'T': fixture}, schema_df, {'k': 'v'})
    back = read_xlsx_table(path, 'T')
assert back.equals(fixture), back
print('round trip unchanged:', len(texts), 'texts')

### 3b. CSVW conversion: empty key component, empty cell, duplicate key

A fixture module with one table, key (K1, K2). An empty K2 gives an empty path segment; an empty cell gives no triple; two rows with the same key stop the conversion.

In [ ]:
FIXTURE_MODULE = """
id: https://kerfors.github.io/cdisc-for-all/def/fixture
name: fixture
prefixes:
  linkml: https://w3id.org/linkml/
imports:
  - linkml:types
default_range: string
annotations:
  source: fixture
classes:
  "T 1":
    class_uri: https://kerfors.github.io/cdisc-for-all/def/fixture#T%201
    unique_keys:
      row_key:
        unique_key_name: row_key
        unique_key_slots:
          - K1
          - K2
    attributes:
      K1:
        slot_uri: https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#K1
      K2:
        slot_uri: https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#K2
      "A (x/y)":
        slot_uri: https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#A%20%28x%2Fy%29
"""
with tempfile.TemporaryDirectory() as tmp:
    mpath = Path(tmp) / 'fixture.yaml'
    mpath.write_text(FIXTURE_MODULE, encoding='utf-8')
    fsv = load_module(mpath)
    check_module(fsv)
    fmeta = csvw_metadata(fsv, 'T 1', 'T%201.csv')
    fixture_shacl = Graph().parse(data=ShaclGenerator(str(mpath)).serialize(), format='turtle')
table = pd.DataFrame([['a', '', 'v'], ['b', 'c d', '']], columns=['K1', 'K2', 'A (x/y)'], dtype=str)
g = table_to_graph(fmeta, table)
for line in canonical_lines(g):
    print(line)
assert URIRef('https://kerfors.github.io/cdisc-for-all/id/fixture/T%201/a/') in set(g.subjects())
assert len(g) == 2 + 4
try:
    table_to_graph(fmeta, pd.concat([table, table.iloc[:1]], ignore_index=True))
    raise AssertionError('duplicate key not stopped')
except ValueError as e:
    print('stopped:', e)

### 3c. Structural SHACL catches a fault

The fixture graph conforms; with a second value in one column and a property that is not a column, it has two Violations.

In [ ]:
ok, _, _ = pyshacl.validate(g, shacl_graph=fixture_shacl)
assert ok
s = URIRef('https://kerfors.github.io/cdisc-for-all/id/fixture/T%201/a/')
bad = Graph()
for t in g:
    bad.add(t)
bad.add((s, URIRef('https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#A%20%28x%2Fy%29'), Literal('w')))
bad.add((s, URIRef('https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#Other'), Literal('z')))
ok, report, _ = pyshacl.validate(bad, shacl_graph=fixture_shacl)
SH = 'http://www.w3.org/ns/shacl#'
violations = set(report.subjects(URIRef(SH + 'resultSeverity'), URIRef(SH + 'Violation')))
assert not ok and len(violations) == 2
print('violations:', len(violations))

### 3d. Canonical N-Triples against a canonicalisation library

`cfa.graph` writes canonical N-Triples itself (no blank nodes, so no relabelling). Compared with pyld's URDNA2015 (the algorithm RDFC-1.0 standardises; pyld does not offer it under the new name) on literals with tab, CR, LF, quote, backslash and non-ASCII text.

In [ ]:
lit = Graph()
for i, text in enumerate(['tab\there', 'cr\r\nlf', 'quote " and \\ backslash', 'ö é ≥ µ', ' ']):
    lit.add((URIRef(f'https://kerfors.github.io/cdisc-for-all/id/fixture/T%201/{i}/'),
             URIRef('https://kerfors.github.io/cdisc-for-all/def/fixture/T%201#A%20%28x%2Fy%29'), Literal(text)))
ours = ''.join(line + '\n' for line in canonical_lines(lit))
theirs = jsonld.normalize(ours, {'algorithm': 'URDNA2015', 'inputFormat': 'application/n-quads',
                                 'format': 'application/n-quads'})
assert ours == theirs, (ours, theirs)
print('canonical N-Triples equal to pyld URDNA2015; graph SHA-256', graph_sha256(lit))